In [ ]:
import re
import pandas as pd
from collections import Counter
from tqdm import tqdm

# L-M eng

## BoW Dict

In [ ]:
lm = pd.read_excel(r'../../Data/Processed_data/Loughran-McDonald_MasterDictionary_1993-2024.xlsx')

In [ ]:
# 2. Build lowercased word sets for each category
cats = ["Positive","Negative","Uncertainty"]
lex = {c.lower(): set(lm.loc[lm[c]>0, "Word"].str.lower()) for c in cats}

# 3. Tokenizer: lowercase + alphabetic tokens
def tokenize(text):
    return re.findall(r"\b[a-z']+\b", text.lower())

# 4. Score function: returns raw counts and proportions
def lm_score(text):
    toks = tokenize(text)
    cnt = Counter(toks)
    N = len(toks) or 1
    return {f"{cat}_count": sum(cnt[w] for w in lex[cat])
            if cat in lex else 0
            for cat in lex} | \
           {f"{cat}_prop": (sum(cnt[w] for w in lex[cat]) / N)
            for cat in lex}

## Classification

In [ ]:
All_news_Bow_processed = pd.read_excel(r'../../Data/All_news_Bow_processed.xlsx').dropna(subset='translation')

In [ ]:
results = []
for idx, row in tqdm(All_news_Bow_processed.iterrows(), total=len(All_news_Bow_processed), desc="BoW scoring"):
    score = lm_score(row['translation'])
    results.append(score)


In [ ]:
# Convert results (list of dicts) to a DataFrame
new_cols = pd.DataFrame(results)

# Concatenate the new columns to your original DataFrame (resetting indices for safe concatenation)
All_news_Bow_processed = pd.concat([All_news_Bow_processed.reset_index(drop=True),
                                     new_cols.reset_index(drop=True)], axis=1)

In [ ]:
All_news_Bow_processed["sentiment"] = All_news_Bow_processed.apply(
    lambda row: max(
        [("Neutral", row["uncertainty_count"]),
         ("Positive", row["positive_count"]),
         ("Negative", row["negative_count"])
         ],
        key=lambda x: x[1]
    )[0],
    axis=1
)

In [ ]:
All_news_Bow_processed.to_excel(r'../../Data/All_news_BoW_processed.xlsx', index=False, engine='xlsxwriter', engine_kwargs={'options': {'strings_to_urls': False}})

# RuSentiLex

## BoW Dict

In [ ]:
dict = pd.read_csv(r"../../Data/Processed_data/rusentilex.txt")[['word', 'tone']]
dict['tone'] =dict["tone"].astype(str).str.lstrip().str.capitalize()

In [ ]:
# 2. Build lowercased word sets for each category
cats = ["Positive","Negative","Neutral"]
lex = {c.lower(): set(dict.loc[dict['tone']==c, "word"].str.lower()) for c in cats}

# 3. Tokenizer: lowercase + alphabetic tokens
def tokenize(text):
    return re.findall(r"\b[а-яёa-z']+\b", text.lower())

# 4. Score function: returns raw counts and proportions
def dict_score(text):
    toks = tokenize(text)
    cnt = Counter(toks)
    N = len(toks) if len(toks) > 0 else 1
    count_dict = {f"{cat}_count": sum(cnt[w] for w in lex[cat])
                  for cat in lex}
    prop_dict = {f"{cat}_prop": sum(cnt[w] for w in lex[cat]) / N
                 for cat in lex}
    return {**count_dict, **prop_dict}

## Classification

In [ ]:
All_news_BoW_ru_processed = pd.read_excel(r'../../Data/All_news_BoW_processed.xlsx').iloc[:,:5]

In [ ]:
results = []
for idx, row in tqdm(All_news_BoW_ru_processed.iterrows(), total=len(All_news_BoW_ru_processed), desc="BoW scoring"):
    score = dict_score(row['news_content'])
    results.append(score)


In [ ]:
# Convert results (list of dicts) to a DataFrame
new_cols = pd.DataFrame(results)

# Concatenate the new columns to your original DataFrame (resetting indices for safe concatenation)
All_news_BoW_ru_processed = pd.concat([All_news_BoW_ru_processed.reset_index(drop=True),
                                     new_cols.reset_index(drop=True)], axis=1)

In [ ]:
All_news_BoW_ru_processed["sentiment"] = All_news_BoW_ru_processed.apply(
    lambda row: max(
        [("Neutral", row["neutral_count"]),
         ("Positive", row["positive_count"]),
         ("Negative", row["negative_count"])
         ],
        key=lambda x: x[1]
    )[0],
    axis=1
)

In [ ]:
All_news_BoW_ru_processed.to_excel(r'../../Data/All_news_BoW_ru_processed.xlsx', index=False, engine='xlsxwriter', engine_kwargs={'options': {'strings_to_urls': False}})